In [ ]:
#https://huggingface.co/speakleash/Bielik-11B-v2.6-Instruct
# ===============================
# 1. Instalacje
# ===============================
!pip install -U bitsandbytes sentence-transformers faiss-cpu transformers accelerate

In [2]:
# ===============================
# 2. Importy
# ===============================
import torch
import numpy as np
import faiss
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer

In [ ]:
# ===============================
# 3. Model językowy – Qwen2.5
# ===============================
#PLLuM i Polish llama wymagala logowania, Qwen jest otwarty, stabilny, dziala w RAG i dobrze obsluguje polski
model_name = "Qwen/Qwen2.5-7B-Instruct"
"""speakleash/Bielik-11B-v2.6-Instruct-bnb-4bit"""

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
# ===============================
# 4. Dokument źródłowy (REGULAMIN)
# ===============================
document = """WYKLAD"""

In [ ]:
# ===============================
# 5. Chunking – każdy slajd to osobny chunk
# ===============================
def chunk_text(text):
    # dzielimy po slajdach z prezentacji, dla lepszego kontekstu
    chunks = [line.strip() for line in text.split("StopkaSlajdu") if line.strip()]
    return chunks

chunks = chunk_text(document)

In [ ]:
# ===============================
# 6. Embeddingi (polski + multi)
# ===============================
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

chunk_embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True
)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(chunk_embeddings))

In [7]:
# ===============================
# 7. Retrieval
# ===============================
def retrieve_context(query, k=3): # k -> 1 - bardzo precyzyjne, 3 - zbalansowane, 6-8 - szum i halucynacje
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb), k)
    return "\n\n".join([chunks[i] for i in indices[0]])

In [8]:
# ===============================
# 8. Funkcja ask_bot
# ===============================
def ask_bot(question):
    # pobierz kontekst z FAISS
    context = retrieve_context(question)

    # print pytania
    print("-" * 80)
    print("PYTANIE:")
    print(question)
    print("-" * 40)

    # print kontekstu
    print("KONTEKST:")
    if context.strip():
        print(context)
    else:
        print("Brak pasujących fragmentów.")
    print("-" * 40)

    # prompt RAG
    prompt = f"""
      Odpowiadaj wyłącznie na podstawie KONTEKSTU.

      ZASADY:
      - Jeżeli KONTEKST zawiera informację potrzebną do odpowiedzi: odpowiedz jednym krótkim zdaniem.
      - Jeżeli KONTEKST NIE zawiera odpowiedzi: odpowiedz dokładnie: "Brak informacji w dokumencie"

      NIE DOPISUJ niczego więcej.
      NIE POWTARZAJ odpowiedzi.
      Nie pokazuj procesu rozumowania.
      Nie dodawaj zadnych komentarzy ani metatekstu.
      Podaj tylko końcową odpowiedź.

      KONTEKST:
      {context}

      PYTANIE:
      {question}

      ODPOWIEDŹ: (krótko, w jednej linijce, bez nowych linii)"""

    # generowanie odpowiedzi
    output = generator(
        prompt,
        max_new_tokens=100, #70 poczatkowo
        temperature=0.05, #0.0-0.1 stricte z dokumentu, 0,2-0.5 - luzno, 0,6-1 - halucynacje, poczatkowo 0.01
        do_sample=False,
        return_full_text=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id #stop sequence, model jest przygotowany pod rozmowe, nie pod jednorazowe pytanie
    )[0]["generated_text"]

    # odcinamy prompt i bierzemy tylko pierwszą linijkę odpowiedzi
    answer = output.split("ODPOWIEDŹ:")[-1].strip()
    answer = answer.split("\n")[0].strip()
    answer = output.split("Human:")[0].strip()

    # print odpowiedzi
    print("ODPOWIEDŹ:")
    print(answer)
    print("-" * 80)

    return


In [ ]:
# ===============================
# 9. Testy
# ===============================
ask_bot("Co to ergonomia pracy?")
ask_bot("Jaki jest cel audytu projektu informatycznego?")
ask_bot("Czym są asocjacje?")

In [10]:
# ===============================
# 10. Testy z uzytkownikiem
# ===============================

asked = ""
while True:
  asked = input("Pytanie od uzytkownika: ")
  if asked == "Q" or asked == "q":
    break
  else:
    ask_bot(asked)
